In [1]:
import pandas as pd

df_train = pd.read_parquet("../Data/yellow_tripdata_2023-01.parquet")
df_test = pd.read_parquet("../Data/yellow_tripdata_2023-02.parquet")

In [2]:
print(len(df_train.columns))
print(df_train.columns.tolist())

19
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [3]:
import statistics as st

df_train["duration"] = (
  df_train["tpep_dropoff_datetime"] - df_train["tpep_pickup_datetime"]).dt.total_seconds() / 60

print(st.stdev(df_train["duration"]))

df_test["duration"] = (
  df_test["tpep_dropoff_datetime"] - df_test["tpep_pickup_datetime"]).dt.total_seconds() / 60

42.59435124195457


In [4]:
print(len(df_train))

3066766


In [5]:
df_train = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)]
df_test = df_test[(df_test.duration >= 1) & (df_test.duration <= 60)]
print((len(df_train) / 3066766) * 100)

98.1220282212598


In [6]:
from sklearn.feature_extraction import DictVectorizer


categorical = ["PULocationID", "DOLocationID"]
numerical = []

df_train[categorical] = df_train[categorical].astype(str)
df_test[categorical] = df_test[categorical].astype(str)

train_dicts = df_train[categorical + numerical].to_dict(orient="records")
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

test_dicts = df_test[categorical + numerical].to_dict(orient="records")
X_test = dv.transform(test_dicts)



In [7]:
X_train.shape

(3009173, 515)

In [8]:
y_train = df_train["duration"].values
y_test = df_test["duration"].values

In [9]:
import numpy as np

for X in (X_train, X_test):
  X.indices = X.indices.astype(np.int32)
  X.indptr = X.indptr.astype(np.int32)

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)
rmse = root_mean_squared_error(y_train, y_pred)
print(f"RMSE on training set: {rmse}")

RMSE on training set: 7.649261931101322


In [11]:
y_pred_test = lr.predict(X_test)
rmse_test = root_mean_squared_error(y_test, y_pred_test)
print(f"RMSE on test set: {rmse_test}")

RMSE on test set: 7.811818553323044
